# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import pprint

# Define the dataset Croissant schema URL
url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset (this will automatically process all Croissant metadata)
dataset = mlc.Dataset(url)

metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\nDescription:\n{metadata.description}\n")


## 2. Data Overview
List available record sets and their fields using their Croissant `@id`s.

In [ ]:
# List all record sets in the dataset using their `@id`
print("Available record sets in the dataset:")
record_sets = []
for rs in dataset.record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '')}")
    record_sets.append(rs['@id'])

if not record_sets:
    print("No record sets found in metadata.\nAttempting to list available records using dataset.records()...")

    # Try to list all available records (commonly there's a default record set with id 'records')
    try:
        records_preview = []
        for i, rec in enumerate(dataset.records()):
            if i < 3:
                records_preview.append(rec)
            else:
                break
        if records_preview:
            print("\nSample records (first 3 rows):")
            for rec in records_preview:
                pprint.pprint(rec)
        else:
            print("No sample records found.")
    except Exception as e:
        print("Error reading sample records:", e)
else:
    # Loop through fields for each record set
    for record_set_id in record_sets:
        print(f"\nFields for record set {record_set_id}:")
        fields = dataset.fields(record_set=record_set_id)
        for fld in fields:
            print(f"  - Field: {fld['@id']}  Name: {fld.get('name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Attempt to extract data from record sets. If there are no explicit record sets, fall back to default.
import collections

dataframes = {}

if not record_sets:
    # If no record sets exist, dataset.records() should return the main data
    print("\nLoading data from default records (no explicit record_set in schema)...")
    main_records = list(dataset.records())
    df_main = pd.DataFrame(main_records)
    dataframes['records'] = df_main
    print("Columns:", df_main.columns.tolist())
    df_main.head()  # Show first 5 rows
else:
    for record_set_id in record_sets:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"\nRecord set: {record_set_id}  Columns: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter records, normalize numeric fields, and group data by key attributes as preparation for further analysis. All references to columns/fields are by their `@id` when possible.

In [ ]:
# For demonstration, select common fields in the survey dataset
df = None
if 'records' in dataframes:
    df = dataframes['records']
else:
    # Use the first available record set
    rs_id = record_sets[0]
    df = dataframes[rs_id]

print("Available columns in DataFrame:")
print(df.columns.tolist())

# Try to use the standard field IDs for GAD-7, PHQ-9, etc, or select a numeric field present
# If unsure, pick 'gad7_total', 'phq9_total', or their @id forms (Croissant often uses IDs like 'gad7_total', 'phq9_total', or URLs)
numeric_field = None
for cand in ['gad7_total', 'GAD-7', 'phq9_total', 'PHQ-9', 'psq_score', 'PSQ', 'age']:
    if cand in df.columns:
        numeric_field = cand
        break

if numeric_field is None:
    raise ValueError("Could not find a suitable numeric field (like 'gad7_total', 'phq9_total', or 'age') in the columns.")

print(f"Selected numeric field for EDA: {numeric_field}")

# Filtering: only include respondents with score > 10
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} respondents")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping: try to group by a demographic field
group_field = None
for cand in ['gender', 'Gender', 'level_of_education', 'Level of Education', 'village', 'Village']:
    if cand in df.columns:
        group_field = cand
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field} (mean):")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize distributions or relationships in the dataset (histogram, boxplot, group comparison, etc).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of selected numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field], bins=10, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If group_field exists, show boxplot
if group_field:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.ylabel(numeric_field)
    plt.xlabel(group_field)
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya. We loaded the Croissant metadata, extracted records, previewed the structure, filtered and normalized key indicator fields (such as GAD-7/PHQ-9 scores), grouped by demographic attributes, and visualized distributions. This illustrates standards-based, reproducible AI dataset exploration using Croissant `@id` references for all major data elements.

For further analyses, consult the [Croissant schema](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) to reference other record sets, fields, and their attributes.